# Dormant 3+ Year Reactivation Personalization — Power Analysis (v2: session/UTM-attributed metric)

**Experiment:** Personalization vs. No Personalization for Dormant 3+ Year Reactivation Campaigns (A/B) · **Owner:** Sergio Oyola  
**Primary metric:** Session/UTM-attributed Reactivation Rate · **Randomization unit:** `client_id` · **Allocation point:** campaign send, restricted to the Dormant 3+ yrs population

This notebook sizes the experiment using a **session/UTM-attributed** reactivation metric: a client only counts as reactivated if there's click-through evidence (a session carrying Blueshift UTM parameters matching the send) linking that specific send to a real demand event. This is a stricter, more conservative standard than a simpler time-proximity-based metric (any qualifying purchase within a fixed window of the send, regardless of channel) — a second candidate metric definition also being evaluated for this experiment. The two trade off differently: time-proximity risks false positives (unrelated coincidental purchases), while session/UTM-attribution risks false negatives (real conversions through untracked paths, e.g. app-only usage with no UTM). Which definition is right for this experiment is the open question this sizing exercise is meant to help resolve.


## Eligibility & Population

The eligible population is clients currently in the **Dormant 3+ years** lifecycle bucket — no checkout in more than **1095 days (3 years)**. This sits on top of the standard lifecycle states already tracked in `curated.checkout_based_client_state_journal`:

| Bucket | Definition |
|---|---|
| Active | Last checkout ≤ 120 days ago |
| Lapsed | Last checkout 120–365 days ago |
| Dormant | Last checkout > 365 days ago |
| **Dormant 3+ yrs (eligible pool)** | Last checkout > 1095 days ago — a subset of Dormant |

`client_state_detail` on the journal table (values `Engaged` / `Lapsed` / `Dormant` / `Never Active`) already implements the Active/Lapsed/Dormant boundaries. The **1095-day cut has no pre-built bucket**, so it's derived by layering `days_since_last_checkout` (from `last_buyable_checkout_ts`) on top of `client_state_detail = 'Dormant'`.


## A/B Design Choices
- **2 arms** (Control = No Personalization / Treatment = Personalization), equal **50/50 split**.
- **Single planned comparison** (Treatment vs. Control); `alpha = 0.05`.
- **Two-sided** test: sizing for a +MDE gives symmetric power to detect harm of the same magnitude, so the Early Stop / harm threshold mirrors the committed MDE (same magnitude, opposite sign).

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)

# Data parameters
START_DATE = '2026-01-01'
SESSION_ATTRIBUTION_WINDOW_DAYS = 7  # a session must occur within this many days of the send to be
                                      # considered a match; generous on purpose since the UTM match
                                      # itself is already a strong identity signal

# Lifecycle thresholds (days since last checkout), per client_state_detail on curated.checkout_based_client_state_journal
ACTIVE_MAX_DAYS = 120
LAPSED_MAX_DAYS = 365
DORMANT_3YR_MIN_DAYS = 1095  # eligibility cutoff for THIS experiment

# Design parameters
INITIAL_ALPHA = 0.05
N_COMPARISONS = 1  # single comparison: Treatment (Personalization) vs. Control (No Personalization)
ALPHA = INITIAL_ALPHA / N_COMPARISONS  # = 0.05, no Bonferroni needed for a single pairwise A/B
POWER = 0.80
TWO_SIDED = True
N_ARMS = 2
MDE_GRID = [0.03, 0.05, 0.10, 0.15]  # relative lift on reactivation rate
HARM_GRID = [-0.03, -0.05]  # relative drop (Early Stop Condition)

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/traitlets/traitlets.py", line

## Step 0 — Confirm lifecycle buckets against the warehouse table

Sanity check `client_state_detail` on `curated.checkout_based_client_state_journal` (current row per client, `is_current = 1`) against the `ACTIVE_MAX_DAYS` / `LAPSED_MAX_DAYS` / `DORMANT_3YR_MIN_DAYS` thresholds, then get the eligible population size for this experiment.

In [2]:
lifecycle_bucket_query = f"""--sql
WITH journal AS (
    SELECT
        client_id,
        client_state_detail,
        date_diff('day', date(last_buyable_checkout_ts), current_date) AS days_since_last_checkout
    FROM curated.checkout_based_client_state_journal
    WHERE is_current = 1
)
SELECT
    CASE
        WHEN client_state_detail = 'Never Active' THEN 'Never Active'
        WHEN days_since_last_checkout <= {ACTIVE_MAX_DAYS} THEN 'Active'
        WHEN days_since_last_checkout <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
        WHEN days_since_last_checkout <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
        ELSE 'Dormant 3+ yrs'
    END AS lifecycle_bucket,
    COUNT(*) AS n_clients
FROM journal
GROUP BY 1
ORDER BY n_clients DESC
"""

lifecycle_bucket_df = query(lifecycle_bucket_query)
lifecycle_bucket_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,lifecycle_bucket,n_clients
0,Never Active,36863824
1,Dormant 3+ yrs,9805722
2,Dormant,2102284
3,Active,1464027
4,Lapsed,868517


The eligible pool for this experiment (**Dormant 3+ yrs, ~9.8M clients**) is large enough; population size is not a constraint.

## Step 1 — Reactivation baseline & eligible daily volume (session/UTM-attributed)

Primary metric = **Reactivation** := the client visited via a session carrying Blueshift UTM parameters matching the send, and that session is linked (via `active_session_id`) to a `curated.client_reactivation_demand_events` row with `demand_type IN ('fix', 'direct_buy')`.

This requires three joins, each doing a distinct job:
1. **Point-in-time eligibility**: a client's Dormant-3yr status is evaluated **as of the send**, via `curated.checkout_based_client_state_journal`'s `start_timestamp`/`end_timestamp` validity window.
2. **Session/UTM match**: join the send to `curated.user_session_conversion_metrics` on `client_id`, `utm_source = 'blueshift'`, `utm_content = send_utm_content` (confirmed to match exactly between the send-side and session-side UTM fields — verified by direct comparison, not assumed), and the session occurring within `SESSION_ATTRIBUTION_WINDOW_DAYS` of the send. This is what makes the metric **click-through-based** rather than time-proximity-based: a qualifying event with no matching UTM'd session doesn't count, no matter how close in time it happened.
3. **Demand confirmation**: join the matched session's `active_session_id` to `client_reactivation_demand_events` to confirm it actually produced a Fix or direct-buy demand event.

**This is expected to show a materially lower baseline rate than a simpler time-proximity-based metric** — not because personalization "performs worse," but because this metric only captures reactivations that route through a *trackable, UTM-tagged click-through session*. Validated separately: only ~1.9% of eligible sends produce any Blueshift-UTM-matched session at all within the window, versus a time-proximity metric which doesn't require a session/click at all. Whether that narrower definition is the *right* one for this experiment — trading time-proximity's false positives (unrelated coincidental conversions) for this metric's false negatives (real conversions via untracked paths, e.g. app-only usage with no UTM) — is the actual decision this comparison is meant to inform.

In [3]:
reactivation_query = f"""--sql
WITH sends AS (
    SELECT
        client_id,
        sent_timestamp,
        send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{START_DATE}'
      AND holdout_group = 0
),
dormant_3yr_sends AS (
    -- point-in-time eligibility: Dormant 3+ yrs status AS OF the send, via the journal's validity window
    SELECT
        s.client_id,
        s.sent_timestamp,
        s.send_utm_content
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
),
matched_sessions AS (
    -- session/UTM match: did the client visit via a session carrying this exact send's UTMs,
    -- within SESSION_ATTRIBUTION_WINDOW_DAYS of the send
    SELECT
        d.client_id,
        d.sent_timestamp,
        u.active_session_id
    FROM dormant_3yr_sends d
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = d.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = d.send_utm_content
        AND u.datetime_in_utc >= d.sent_timestamp
        AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{START_DATE}'
),
attributed_demand AS (
    -- demand confirmation: did that matched session actually produce a fix/direct-buy demand event
    SELECT DISTINCT
        ms.client_id,
        ms.sent_timestamp
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events e
        ON e.active_session_id = ms.active_session_id
    WHERE e.demand_type IN ('fix', 'direct_buy')
),
client_month AS (
    -- collapse to client x month: did THIS client reactivate at all that month
    -- (joined on client_id AND sent_timestamp, not client_id alone, so one attributed
    -- reactivation can't leak into unrelated months for the same client)
    SELECT
        d.client_id,
        DATE_TRUNC('month', d.sent_timestamp) AS month,
        MAX(CASE WHEN a.client_id IS NOT NULL THEN 1 ELSE 0 END) AS client_reactivated
    FROM dormant_3yr_sends d
    LEFT JOIN attributed_demand a
        ON d.client_id = a.client_id AND d.sent_timestamp = a.sent_timestamp
    GROUP BY 1, 2
),
month_days AS (
    SELECT
        DATE_TRUNC('month', sent_timestamp) AS month,
        COUNT(DISTINCT CAST(sent_timestamp AS DATE)) AS days_observed
    FROM dormant_3yr_sends
    GROUP BY 1
)
SELECT
    cm.month,
    md.days_observed,
    COUNT(DISTINCT cm.client_id) AS unique_clients_sent,
    SUM(cm.client_reactivated) AS reactivated_clients,
    CAST(SUM(cm.client_reactivated) AS DOUBLE) / COUNT(DISTINCT cm.client_id) AS client_reactivation_rate,
    ROUND(COUNT(DISTINCT cm.client_id) * 1.0 / md.days_observed, 1) AS unique_clients_per_day
FROM client_month cm
JOIN month_days md ON cm.month = md.month
GROUP BY cm.month, md.days_observed
ORDER BY cm.month DESC
"""

reactivation_df = query(reactivation_query)
reactivation_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,unique_clients_sent,reactivated_clients,client_reactivation_rate,unique_clients_per_day
0,2026-07-01 00:00:00.000,15,1146683,1046,0.000912,76445.5
1,2026-06-01 00:00:00.000,30,1204986,2395,0.001988,40166.2
2,2026-05-01 00:00:00.000,31,1243651,3034,0.002440,40117.8
3,2026-04-01 00:00:00.000,30,1198458,3505,0.002925,39948.6
4,2026-03-01 00:00:00.000,31,1196460,4529,0.003785,38595.5
5,2026-02-01 00:00:00.000,28,1188131,3577,0.003011,42433.3
6,2026-01-01 00:00:00.000,31,1338048,4135,0.003090,43162.8


## Step 1.1 — Data sources and reprocessing lag

The reprocessing behavior of `blueshift.campaign_activity_kpis` (the table behind a time-proximity-based metric) is confirmed against its actual pipeline code (`kpi_join.py` + the `de_blueshift` DAG): a 45-day window after which flags freeze, and rates measured off that table do flatten out once every send in a month has cleared the window. **No equivalent confirmation exists yet for `curated.user_session_conversion_metrics` or `curated.client_reactivation_demand_events`** — neither table's ETL/DDL is visible in this codebase, and no table owner has confirmed a reprocessing window for either one.

The observed monthly rate here does decline as months get more recent (see `reactivation_df` above) — the shape you'd expect from a data-lag artifact. But unlike the confirmed 45-day case, this decline **doesn't clearly flatten out** for months that are already well past 45 days old (e.g. March and April both have 45+ days elapsed as of this analysis and are still meaningfully different from each other) — so either this pipeline has a longer/different reprocessing window than kpis, or part of the decline is a genuine month-to-month trend (campaign mix, seasonality) rather than a data-lag artifact.

**Given that uncertainty, this analysis conservatively uses the oldest available month as `REFERENCE_MONTH`** (maximizing elapsed time regardless of the unknown window). This trades staleness/seasonality risk for safety against under-counting.

In [4]:
reactivation_df['month_ts'] = pd.to_datetime(reactivation_df['month'])

# Conservative choice pending confirmation from the table owners (see Step 1.1): use the oldest
# available month, not the most recent, since we can't yet confirm how long this pipeline takes
# to fully settle.
REFERENCE_MONTH = reactivation_df.sort_values('month_ts', ascending=True).iloc[0]

BASELINE_RATE = float(REFERENCE_MONTH['client_reactivation_rate'])
DAILY_ELIGIBLE_CLIENTS = float(REFERENCE_MONTH['unique_clients_per_day'])

print(f"REFERENCE_MONTH = {REFERENCE_MONTH['month']}  (oldest available month -- conservative choice, no confirmed maturity gate for this pipeline)")
print(f"BASELINE_RATE = {BASELINE_RATE:.4%}  |  DAILY_ELIGIBLE_CLIENTS = {DAILY_ELIGIBLE_CLIENTS:,.0f}")
reactivation_df

REFERENCE_MONTH = 2026-01-01 00:00:00.000  (oldest available month -- conservative choice, no confirmed maturity gate for this pipeline)
BASELINE_RATE = 0.3090%  |  DAILY_ELIGIBLE_CLIENTS = 43,163


,month,days_observed,unique_clients_sent,reactivated_clients,client_reactivation_rate,unique_clients_per_day,month_ts
0,2026-07-01 00:00:00.000,15,1146683,1046,0.000912,76445.5,2026-07-01
1,2026-06-01 00:00:00.000,30,1204986,2395,0.001988,40166.2,2026-06-01
2,2026-05-01 00:00:00.000,31,1243651,3034,0.002440,40117.8,2026-05-01
3,2026-04-01 00:00:00.000,30,1198458,3505,0.002925,39948.6,2026-04-01
4,2026-03-01 00:00:00.000,31,1196460,4529,0.003785,38595.5,2026-03-01
5,2026-02-01 00:00:00.000,28,1188131,3577,0.003011,42433.3,2026-02-01
6,2026-01-01 00:00:00.000,31,1338048,4135,0.003090,43162.8,2026-01-01


## Step 2 — Sample size & duration

`n_total_statsmodels` is pairwise, which is exactly what a 2-arm A/B needs. `n_treatment` **is** the per-arm requirement (50/50 split), `total = 2 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE_CLIENTS / 2` per day. This metric's low baseline rate (session/UTM-attributed reactivation is rare, by design) means larger samples and longer durations are needed here than a broader, less strict metric definition would require for the same relative MDE.

In [5]:
def size_table(rel_grid, baseline, daily, label):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[0.5],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_2arm'] = df['n_per_arm'] * N_ARMS
    if daily:
        df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
        df['weeks_required'] = (df['days_required'] / 7).round(1)
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_2arm','days_required','weeks_required']
    else:
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_2arm']

    sided = 'two-sided' if TWO_SIDED else 'one-sided'
    print(f"--- {label} (baseline={baseline:.2%}, alpha={ALPHA} (no multiple-comparison correction), power={POWER:.0%}, {sided}) ---")

    return df[cols]

size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE_CLIENTS, 'Positive MDE')

--- Positive MDE (baseline=0.31%, alpha=0.05 (no multiple-comparison correction), power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,+3%,0.003183,5710733,11421466,265,37.9
1,+5%,0.003245,2076054,4152108,97,13.9
2,+10%,0.003399,531631,1063262,25,3.6
3,+15%,0.003554,241887,483774,12,1.7


In [6]:
# Harm side (Early Stop Condition). With a two-sided test these n's mirror the positive grid;
# shown explicitly to document the harm magnitude the test is powered to detect.
size_table(HARM_GRID, BASELINE_RATE, DAILY_ELIGIBLE_CLIENTS, 'Harm / guardrail')

--- Harm / guardrail (baseline=0.31%, alpha=0.05 (no multiple-comparison correction), power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,-3%,0.002998,5542459,11084918,257,36.7
1,-5%,0.002936,1975089,3950178,92,13.1


## Step 3 — Summary for Experiment Design doc

The MDE should be defined in advance, then pasted to document the Power Analysis section in the Experiment Design doc. Before pasting this in, make sure this is the metric definition you're actually committing to for the experiment — if another candidate metric is also under evaluation, don't mix summaries from different metric definitions into the same design doc.

In [7]:
TARGET_REL_MDE = 0.10  # NOTE: placeholder relative MDE (10% relative lift on reactivation rate) pending Product sign-off

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[0.5],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED
    )

n_per_arm = int(list(res.values())[0]['n_treatment'])

duration = f"{int(np.ceil(n_per_arm / (DAILY_ELIGIBLE_CLIENTS/N_ARMS)))} days" if DAILY_ELIGIBLE_CLIENTS else 'TODO (run Step 1)'

summary = {
    'Metric Used': f"Session/UTM-attributed Reactivation Rate (Fix OR Direct Buy demand event, linked via a Blueshift-UTM-matched session within {SESSION_ATTRIBUTION_WINDOW_DAYS} days of send)",
    'Population': 'Dormant 3+ yrs clients (no checkout in >1095 days), campaign-eligible',
    'Baseline Value': f"{BASELINE_RATE:.2%} client reactivation rate ({REFERENCE_MONTH['month']}, oldest available month -- conservative pending confirmation of this pipeline's reprocessing behavior, see Step 1.1)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.4f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.4f})",
    'One/Two-Sided Test': 'Two-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction needed)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control: No Personalization / Treatment: Personalization)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': duration,
    'Early Stop / harm threshold': f"-{abs(TARGET_REL_MDE):.0%} relative (covered by two-sided sizing)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,"Session/UTM-attributed Reactivation Rate (Fix OR Direct Buy demand event, linked via a Blueshift-UTM-matched session within 7 days of send)"
Population,"Dormant 3+ yrs clients (no checkout in >1095 days), campaign-eligible"
Baseline Value,"0.31% client reactivation rate (2026-01-01 00:00:00.000, oldest available month -- conservative pending confirmation of this pipeline's reprocessing behavior, see Step 1.1)"
Minimum Detectable Effect,+10% relative (0.0031 -> 0.0034)
One/Two-Sided Test,Two-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction needed)"
Statistical Power,80%
Variant Split %,50% / 50% (Control: No Personalization / Treatment: Personalization)
Minimum Samples by Variant,"531,631"
Minimum Samples total,"1,063,262"
